# Experiment 030: N=4 group-size ablation at 7B (Colab)

One seed-matched arm compares N=4 with the archived 013 seed-0 N=8 endpoint.
Both runs use 500 optimiser steps and the exact same merged GSM8K-SFT starting
model. This is a method-as-practised comparison because N = 4 samples half as
many completions over the run. Hard-tier accuracy is only inferential
endpoint; all conclusions stay scoped to this seed-0 pair.

One Run-all per session. Finished work is checked and skiped on a rerun.


## 0. Compute units and mount Drive


In [ ]:
# record the balance before the session starts
import glob, json, os, time
session_id = globals().get('session_id', time.strftime('%Y%m%d_%H%M%S'))
cu_before = 653.57      # fill from the resources panel
unit_rate = 6.77     # units per hour for this card
print('session', session_id, '| cu_before =', cu_before if cu_before is not None else 'FILL ME')

from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/inm363'
RESULTS = os.path.join(ROOT, 'results')
OPEN_UNITS = os.path.join(RESULTS, 'open_cu_sessions')
UNITS_LOG = os.path.join(RESULTS, 'colab_units.jsonl')
os.makedirs(OPEN_UNITS, exist_ok=True)

# close an unfinished session with the balance captured at this runtime start
old_sessions = [p for p in glob.glob(os.path.join(OPEN_UNITS, '*.json'))
                if not p.endswith(f'{session_id}.json')]
open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
if cu_before is None:
    print('reminder: fill cu_before, then rerun this cell to save the session')
else:
    for old_path in old_sessions:
        with open(old_path) as f:
            old = json.load(f)
        old['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        old['colab_cu_after'] = cu_before
        old['colab_cu_used'] = round(old['colab_cu_before'] - cu_before, 2)
        old['closed_on_resume'] = True
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(old) + '\n')
        os.remove(old_path)
        print('closed previous session:', old['session_id'])

    if not os.path.exists(open_path):
        start = {'session_id': session_id, 'ts_start': time.strftime('%Y-%m-%d %H:%M'),
                 'notebook': '030_colab_b7_n4_ablation',
                 'colab_cu_before': cu_before, 'unit_rate': unit_rate}
        with open(open_path, 'w') as f:
            json.dump(start, f)
        print('opened session record')


session 20260812_141613 | cu_before = 653.57
Mounted at /content/drive
opened session record


## 1. GPU check: A100 with 80GB required


In [ ]:
import subprocess, torch
gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
print(gpu.strip() or 'no gpu visible')
total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'total vram: {total_gb:.1f} GiB')
assert 'A100' in gpu and total_gb >= 79, 'need the a100 80gb runtime used by the 013 reference arm'


GPU 0: NVIDIA A100-SXM4-80GB (UUID: GPU-1ff77368-ec22-4d8b-ad8b-1317727adfba)
total vram: 79.3 GiB


## 2. Clone or update the repo


In [ ]:
import base64
import getpass
import glob
import os
from pathlib import Path
import subprocess

REPO_SLUG = 'hastov-lab/rl-grpo-or-sft-for-llms'
repo_url = f'https://github.com/{REPO_SLUG}.git'
repo_root = Path('/content') / REPO_SLUG.split('/')[-1]


def repo_git(*args):
    git = ['git']
    env = None
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None

    # Private forks can use a read token from Colab Secrets.
    if userdata is not None and REPO_SLUG != 'hastov-lab/rl-grpo-or-sft-for-llms':
        try:
            token = userdata.get('GH_TOKEN')
        except Exception:
            token = None
        if isinstance(token, dict):
            token = token.get('value')
        if not isinstance(token, str) or not token.strip():
            token = getpass.getpass('GitHub token: ')
        if not token.strip():
            raise RuntimeError('This private fork requires a GitHub token with read access.')

        # Keep the token out of the command line and the saved remote URL.
        auth = base64.b64encode(f'x-access-token:{token.strip()}'.encode()).decode()
        env = dict(os.environ, REPO_GIT_AUTH=f'Authorization: Basic {auth}')
        git.append('--config-env=http.https://github.com/.extraHeader=REPO_GIT_AUTH')

    subprocess.run([*git, *args], env=env, check=True)


if repo_root.exists():
    if not (repo_root / '.git').exists():
        raise RuntimeError(f'Expected a Git checkout at {repo_root}.')
    origin = subprocess.check_output(
        ['git', '-C', str(repo_root), 'config', '--local', '--get', 'remote.origin.url'], text=True
    ).strip()
    if origin != repo_url:
        raise RuntimeError(f'The checkout at {repo_root} belongs to a different repository.')
    branch = subprocess.check_output(
        ['git', '-C', str(repo_root), 'branch', '--show-current'], text=True
    ).strip()
    if branch != 'main':
        raise RuntimeError(f'Expected the main branch at {repo_root}.')
    repo_git('-C', str(repo_root), 'pull', '--ff-only', 'origin', 'main')
else:
    repo_git('clone', '--branch', 'main', '--single-branch', '--depth', '1', repo_url, str(repo_root))

# Use the scripts from this checkout even if another experiment is open.
code_root = repo_root / '001_Code'
if not (code_root / 'scripts' / 'run.py').is_file():
    raise RuntimeError(f'Experiment scripts are missing from {code_root}.')
os.chdir(code_root)
print('working in:', os.getcwd())


Cloning into 'INM363-IndividualProject-Dissertation'...
remote: Enumerating objects: 3777, done.
remote: Counting objects: 100% (103/103), done.
remote: Compressing objects: 100% (78/78), done.
remote: Total 3777 (delta 31), reused 83 (delta 25), pack-reused 3674 (from 1)
Receiving objects: 100% (3777/3777), 113.49 MiB | 19.96 MiB/s, done.
Resolving deltas: 100% (2430/2430), done.
Updating files: 100% (1742/1742), done.
working in: /content/INM363-IndividualProject-Dissertation/001_Code


## 3. Install dependencies


In [ ]:
!pip install -q trl==1.7.1 transformers==5.5.4 peft==0.19.1 accelerate==1.14.0 datasets==4.8.4 wandb pyyaml


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.4/842.4 kB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 161.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 55.0 MB/s eta 0:00:00


## 4. W&B key


In [ ]:
import getpass, os
key = None
try:
    from google.colab import userdata
    key = userdata.get('WANDB_API_KEY')
except Exception:
    key = None
if not key and os.path.exists('wandb_api_key.txt'):
    key = open('wandb_api_key.txt').read().strip()
if not key:
    key = getpass.getpass('wandb api key: ').strip()
assert key, 'a wandb key is required because the worker logs online'
os.environ['WANDB_API_KEY'] = key
with open('wandb_api_key.txt', 'w') as f:
    f.write(key)
key = None
print('wandb key set')


wandb key set


## 5. Preflight before any test-set contact


In [ ]:
import json, os, shutil, sys
sys.path.insert(0, os.getcwd())
from config import load_config, run_tag
from scripts.s12_eval import atomic_write_json, config_hash_status
from scripts.s28_compose import (check_merged_complete, merged_hashes,
                                 resolve_source_adapter, verify_historical_merge)

D = '/content/drive/MyDrive/inm363'
CFG_PATH = 'configs/030_colab_b7_n4_seed0.yaml'
PARENT_TAG = '013_COLAB_FULL_B7_sft_grpo_seed0'
cfg = load_config(CFG_PATH)
TAG = run_tag(cfg)
assert cfg.experiment == '030' and cfg.train.seed == 0
assert cfg.condition.name == 'EXTERNAL_SFT_GRPO'
assert cfg.train.num_generations == 4

problems = []
try:
    ADIR, ADAPTER_HASHES = resolve_source_adapter(cfg.train.grpo_from_adapter, D)
except Exception as e:
    problems.append(str(e))
    ADIR, ADAPTER_HASHES = None, None

PARENT_RESULTS = os.path.join(D, 'results', PARENT_TAG)
ARCHIVE_MERGED = os.path.join(D, 'checkpoints', PARENT_TAG, 'sft_merged')
HISTORICAL_MERGE_MANIFEST = 'results/022_smooth_B7_seed0_step1000/arm_manifest.json'
if ADIR:
    parent_cfg = load_config('configs/013_colab_full_b7_seed0.yaml')
    parent_man = json.load(open(os.path.join(PARENT_RESULTS, 'manifest.json')))
    stored = open(os.path.join(ADIR, 'config_hash.txt')).read().strip()
    hash_status = config_hash_status(parent_cfg, parent_man, stored)
    if hash_status not in ('current', 'legacy_missing_grpo_from_adapter'):
        problems.append(f'parent adapter hash status is {hash_status}')
else:
    hash_status = None

for name in ('sft_eval.json', 'sft_preds.jsonl', 'sft_logprob.json',
             'sft_logprob_preds.jsonl', 'grpo_eval.json', 'grpo_preds.jsonl',
             'grpo_logprob.json', 'grpo_logprob_preds.jsonl',
             'grpo_trainlog.json', 'manifest.json'):
    if not os.path.exists(os.path.join(PARENT_RESULTS, name)):
        problems.append(f'013 parent missing {name}')
try:
    check_merged_complete(ARCHIVE_MERGED)
    ARCHIVE_HASHES = merged_hashes(ARCHIVE_MERGED)
    HISTORICAL_MERGE = verify_historical_merge(
        ARCHIVE_MERGED, HISTORICAL_MERGE_MANIFEST, PARENT_TAG)
except Exception as e:
    problems.append(str(e))
    ARCHIVE_HASHES = None
    HISTORICAL_MERGE = None

free_gb = shutil.disk_usage(D).free / 1024**3
print(f'drive free: {free_gb:.0f} GB')
if free_gb < 30:
    problems.append(f'only {free_gb:.0f}GB free on drive, the copied 7b base and checkpoints need 30')
assert not problems, 'preflight halt:\n' + '\n'.join(problems)

preflight = {'config': CFG_PATH, 'tag': TAG, 'parent_tag': PARENT_TAG,
             'adapter_sha256': ADAPTER_HASHES,
             'archive_source': ARCHIVE_MERGED,
             'archive_sha256': ARCHIVE_HASHES,
             'historical_merge': HISTORICAL_MERGE,
             'parent_hash_status': hash_status}
pf_path = os.path.join(D, 'results', '030_preflight.json')
if os.path.exists(pf_path):
    assert json.load(open(pf_path)) == preflight, '030 preflight sources changed, halting'
    print('preflight matches the recorded hashes')
else:
    atomic_write_json(pf_path, preflight)
    print('preflight recorded')
print('preflight green:', TAG)


drive free: 165 GB
preflight recorded
preflight green: 030_COLAB_FULL_B7_external_sft_grpo_seed0


## 6. Build the data splits


In [ ]:
!python scripts/build_data.py


README.md: 100% 7.93k/7.93k [00:00<00:00, 22.3MB/s]

main/train-00000-of-00001.parquet: downloading bytes:   2% 54.6k/2.31M [00:01<00:59, 37.9kB/s]
main/train-00000-of-00001.parquet: downloading bytes: 100% 2.30M/2.30M [00:01<00:00, 1.56MB/s,  223kB/s  ]
main/train-00000-of-00001.parquet: reconstructing file: 100% 2.31M/2.31M [00:01<00:00, 1.56MB/s,  224kB/s  ]

main/test-00000-of-00001.parquet: downloading bytes:   0% 0.00/419k [00:00<?, ?B/s]
main/test-00000-of-00001.parquet: downloading bytes: 100% 419k/419k [00:01<00:00, 327kB/s, 40.8kB/s  ]
main/test-00000-of-00001.parquet: reconstructing file: 100% 419k/419k [00:01<00:00, 327kB/s, 40.8kB/s  ]
Generating train split: 100% 7473/7473 [00:00<00:00, 444180.40 examples/s]
Generating test split: 100% 1319/1319 [00:00<00:00, 267389.41 examples/s]
README.md: 100% 53.2k/53.2k [00:00<00:00, 82.7MB/s]
dataset_infos.json: 100% 138k/138k [00:00<00:00, 138MB/s]

high_school_psychology/test-00000-of-000(…): downloading bytes:   0% 0.00/92.8k [00

## 7. Reuse the archived base evaluation


In [ ]:
from scripts.s28_compose import prepopulate_base_evals
src = os.path.join(D, 'results', 'base_evals', '013_COLAB_FULL_B7')
dst = os.path.join(D, 'results', 'base_evals', '030_COLAB_FULL_B7')
base_record = prepopulate_base_evals(src, dst)
print('base reuse covers', sorted(base_record['sha256']))


  base evals prepopulated from 013_COLAB_FULL_B7
base reuse covers ['base_eval.json', 'base_logprob.json', 'base_logprob_preds.jsonl', 'base_preds.jsonl']


## 8. Copy the exact 013 merged starting model


In [ ]:
from scripts.s28_compose import stage_archived_merge, verify_archived_merge
OUT_DIR = os.path.join(D, 'checkpoints', TAG, 'grpo')
MERGED_DIR = os.path.join(D, 'checkpoints', TAG, 'sft_merged')
merge_record = stage_archived_merge(ARCHIVE_MERGED, MERGED_DIR, ADIR, OUT_DIR)
verify_archived_merge(ARCHIVE_MERGED, MERGED_DIR, ADIR)
print('copied files:', len(merge_record['destination_sha256']))


  archived 013 merged base copied and committed
copied files: 6


## 9. Wiring checks


In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, '-m', 'pytest',
                    'tests/test_030_n4_ablation.py', 'tests/test_s28_composition.py',
                    'tests/test_checkpoint.py', 'tests/test_grpo_worker.py', '-q'])
assert r.returncode == 0, 'package tests failed, nothing runs'

from scripts.grpo_worker import (active_worker_processes, blocked_resume_reason,
                                 stream_subprocess)
from scripts.s12_eval import (IntegrityError, identity_check, read_jsonl,
                              sha256, atomic_write_json)
from scripts.s28_compose import (arm_state, commit_endpoint_eval,
                                 commit_training_complete, conclude_a2,
                                 concluded_a2_state, endpoint_state, gate_state,
                                 sentinel_abort_reason, status_is_a2, training_state,
                                 verify_archived_merge, write_arm_complete)
from obs import _versions, append_ledger, gpu_snapshot, _git_info
from train.model_utils import resolve_dtype
print('package imports fine')


package imports fine


In [ ]:
!pip uninstall -q -y torchao


## 10. Parent identity probe


In [ ]:
import shutil as _shutil, subprocess, sys, time

GATE_SCRIPT = '/content/s30_gate.py'
with open(GATE_SCRIPT, 'w') as f:
    f.write('''import argparse, os, sys
sys.path.insert(0, os.getcwd())
from config import load_config
from scripts.env_switch import apply_smoke, resolve_device
from eval.harness import run_logprob_eval

ap = argparse.ArgumentParser()
ap.add_argument("--config", required=True)
ap.add_argument("--adapter", required=True)
ap.add_argument("--out", required=True)
args = ap.parse_args()
cfg = apply_smoke(load_config(args.config))
run_logprob_eval(cfg, resolve_device(cfg.env), adapter_path=args.adapter,
                 condition="gate", out_dir=args.out)
''')

RES_DIR = os.path.join(D, 'results', TAG)
os.makedirs(RES_DIR, exist_ok=True)
gate_path = os.path.join(RES_DIR, 'parent_gate.json')
archived_preds = os.path.join(PARENT_RESULTS, 'sft_logprob_preds.jsonl')
if gate_state(gate_path, ADAPTER_HASHES, archived_preds) == 'committed':
    print('parent gate already committed, skipping')
else:
    gate_dir = '/content/s30_gate_seed0'
    gate_preds = os.path.join(gate_dir, 'gate_logprob_preds.jsonl')
    rows_ok = (os.path.exists(gate_preds)
               and sum(1 for line in open(gate_preds) if line.strip()) == 1884)
    if not rows_ok:
        _shutil.rmtree(gate_dir, ignore_errors=True)
        t0 = time.time()
        r = subprocess.run([sys.executable, GATE_SCRIPT, '--config', CFG_PATH,
                            '--adapter', ADIR, '--out', gate_dir])
        assert r.returncode == 0, 'parent identity probe failed'
        elapsed = round(time.time() - t0, 1)
    else:
        elapsed = None
    new_rows = read_jsonl(gate_preds)
    assert len(new_rows) == 1884, 'identity probe scratch is incomplete'
    stats = identity_check(new_rows, read_jsonl(archived_preds))
    atomic_write_json(gate_path, {
        'ts': time.strftime('%Y-%m-%d %H:%M'), 'completed': True,
        'parent': PARENT_TAG, 'adapter_sha256': ADAPTER_HASHES,
        'archived_preds_sha256': sha256(archived_preds),
        'new_preds_sha256': sha256(gate_preds),
        'stats': stats, 'elapsed_sec': elapsed})
    _shutil.rmtree(gate_dir, ignore_errors=True)
    print('parent gate:', stats)

free_gib = torch.cuda.mem_get_info()[0] / 1024**3
print(f'gpu free: {free_gib:.1f} GiB')
assert free_gib >= 70, 'need 70 GiB free before the 7b training run'


parent gate: {'agreement': 1.0, 'acc_new': 0.800425, 'acc_archived': 0.800425}
gpu free: 78.8 GiB


## 11. Train and evaluate N=4


In [13]:
assert cu_before is not None and unit_rate is not None, 'fill cu_before and unit_rate in the units cell'
import glob as _glob, subprocess, sys, time

EVAL_SCRIPT = '/content/s30_eval.py'
with open(EVAL_SCRIPT, 'w') as f:
    f.write('''import argparse, os, sys
sys.path.insert(0, os.getcwd())
from checkpoint import save_metrics
from config import load_config
from scripts.env_switch import apply_smoke, resolve_device
from eval.harness import run_eval, run_logprob_eval

ap = argparse.ArgumentParser()
ap.add_argument("--config", required=True)
ap.add_argument("--adapter", required=True)
ap.add_argument("--out", required=True)
args = ap.parse_args()
cfg = apply_smoke(load_config(args.config))
device = resolve_device(cfg.env)
ev = run_eval(cfg, device, adapter_path=args.adapter, condition="grpo", out_dir=args.out)
save_metrics(os.path.join(args.out, "grpo_eval.json"), ev)
lp = run_logprob_eval(cfg, device, adapter_path=args.adapter, condition="grpo", out_dir=args.out)
save_metrics(os.path.join(args.out, "grpo_logprob.json"), lp)
''')

os.environ['WANDB_PROJECT'] = 'inm363-grpo'
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
LEDGER = os.path.join(D, 'results', 'compute_ledger.jsonl')
allowed_infra_files = {
    'notebooks/030_colab_b7_n4_ablation.ipynb',
    'scripts/grpo_worker.py',
    'tests/test_030_n4_ablation.py',
    'tests/test_grpo_worker.py',
}

def newest_step():
    checkpoints = _glob.glob(os.path.join(OUT_DIR, 'checkpoint-*'))
    return max((int(p.rsplit('-', 1)[1]) for p in checkpoints), default=0)

def save_available_history():
    path = os.path.join(RES_DIR, 'grpo_trainlog.json')
    if os.path.exists(path):
        return json.load(open(path))
    checkpoints = sorted(_glob.glob(os.path.join(OUT_DIR, 'checkpoint-*')),
                         key=lambda p: int(p.rsplit('-', 1)[1]), reverse=True)
    for checkpoint in checkpoints:
        state_path = os.path.join(checkpoint, 'trainer_state.json')
        if os.path.exists(state_path):
            history = json.load(open(state_path)).get('log_history', [])
            if history:
                atomic_write_json(path, history)
                return history
    return []

parent_manifest = json.load(open(os.path.join(PARENT_RESULTS, 'manifest.json')))
for lib in ('torch', 'trl', 'transformers', 'peft', 'accelerate',
            'cais/mmlu@', 'openai/gsm8k@'):
    want = parent_manifest.get('versions', {}).get(lib)
    if want is not None:
        got = _versions().get(lib)
        assert got == want, f'013 {lib}: {got} vs {want}'
assert resolve_dtype(cfg) == parent_manifest['resolved_dtype'], 'dtype differs from 013'
assert cfg.exec.allow_oom_fallback is False, 'the ladder must stay off'
assert (cfg.exec.grpo_batch_size or cfg.train.num_generations) == 4
assert cfg.exec.grpo_grad_accum == 1 and cfg.exec.gradient_checkpointing is False
assert cfg.log.wandb_project == 'inm363-grpo'
verify_archived_merge(ARCHIVE_MERGED, MERGED_DIR, ADIR)

final_artifacts = [os.path.join(OUT_DIR, 'adapter_model.safetensors'),
                   os.path.join(OUT_DIR, 'adapter_base.json'),
                   os.path.join(RES_DIR, 'grpo_trainlog.json')]
if arm_state(RES_DIR, OUT_DIR) == 'committed':
    assert training_state(RES_DIR, OUT_DIR, MERGED_DIR, cfg) == 'committed'
    verify_archived_merge(ARCHIVE_MERGED, MERGED_DIR, ADIR)
    print('arm complete, skipping')
else:
    man_path = os.path.join(RES_DIR, 'arm_manifest.json')
    man = json.load(open(man_path)) if os.path.exists(man_path) else {'attempts': []}
    start_step = newest_step()
    for attempt in man.get('attempts', []):
        if 'ts_end' not in attempt:
            attempt.update({'interrupted': True, 'end_step': start_step,
                            'elapsed_sec': None, 'ts_end': 'runtime died before finally'})
    atomic_write_json(man_path, man)

    if concluded_a2_state(RES_DIR, man.get('attempts', [])) == 'committed':
        print('arm concluded by sentinel a2 previously, skipping')
    else:
        a2_prior = sentinel_abort_reason(man.get('attempts', []))
        if a2_prior:
            history = save_available_history()
            evidence = {'num_generations': 4, 'history_records': len(history),
                        'last_logged_step': max((x.get('step', 0) for x in history), default=0)}
            conclude_a2(RES_DIR, man['attempts'][-1], evidence=evidence)
            print('repaired the missing a2 conclusion record')
        else:
            reason = blocked_resume_reason(man.get('attempts', []))
            assert reason is None, reason
            assert not active_worker_processes(CFG_PATH), 'a training worker is already running'

            trained = all(os.path.exists(path) for path in final_artifacts)
            if not trained:
                env = {'git': _git_info(), 'versions': _versions(),
                       'resolved_dtype': resolve_dtype(cfg), 'ladder': 'disabled'}
                repo_top = subprocess.run(['git', 'rev-parse', '--show-toplevel'],
                                          capture_output=True, text=True, check=True).stdout.strip()
                repo_prefix = os.path.relpath(os.getcwd(), repo_top)
                for attempt in man.get('attempts', []):
                    previous = attempt.get('env')
                    if not previous:
                        continue
                    old_commit = previous['git'].get('commit')
                    new_commit = env['git'].get('commit')
                    if old_commit != new_commit:
                        changed = subprocess.run(['git', 'diff', '--name-only', old_commit, new_commit],
                                                 capture_output=True, text=True,
                                                 check=True).stdout.splitlines()
                        prefix = repo_prefix + os.sep
                        changed = [name[len(prefix):] if repo_prefix != '.' and name.startswith(prefix)
                                   else name for name in changed]
                        unexpected = sorted(set(changed) - allowed_infra_files)
                        assert not unexpected, f'scientific code drifted between attempts: {unexpected}'
                    for lib in ('torch', 'trl', 'transformers', 'peft', 'accelerate'):
                        assert previous['versions'].get(lib) == env['versions'].get(lib), f'{lib} drifted'
                    assert previous['resolved_dtype'] == env['resolved_dtype'], 'dtype drifted'

                free_gib = torch.cuda.mem_get_info()[0] / 1024**3
                assert free_gib >= 70, f'only {free_gib:.1f} GiB free before training'
                verify_archived_merge(ARCHIVE_MERGED, MERGED_DIR, ADIR)
                attempt = {'start_step': start_step, 'ts_start': time.strftime('%Y-%m-%d %H:%M'),
                           'gpu': gpu_snapshot(), 'env': env,
                           'adapter_sha256': ADAPTER_HASHES, 'completed': False}
                man['attempts'].append(attempt)
                atomic_write_json(man_path, man)

                status_path = '/content/030_worker_seed0.json'
                if os.path.exists(status_path):
                    os.remove(status_path)
                worker_cmd = [sys.executable, '-u', 'scripts/grpo_worker.py',
                              '--config', CFG_PATH, '--adapter', ADIR,
                              '--out-dir', OUT_DIR, '--log-dir', RES_DIR,
                              '--run-name', f'{TAG}_grpo', '--status', status_path,
                              '--report-to', 'wandb']
                t0 = time.time()
                proc = None
                parent_failure = None
                a2_concluded = False
                try:
                    print(f'===== N=4 from step {start_step} =====')
                    proc = stream_subprocess(worker_cmd)
                    if proc.returncode != 0:
                        status_now = json.load(open(status_path)) if os.path.exists(status_path) else {}
                        if status_is_a2(status_now):
                            a2_concluded = True
                        else:
                            raise RuntimeError(f'training worker failed with exit code {proc.returncode}')
                except BaseException as exc:
                    parent_failure = {'error_type': type(exc).__name__,
                                      'error_message': str(exc)[:2000]}
                    raise
                finally:
                    wall = time.time() - t0
                    end_step = newest_step()
                    worker_returncode = proc.returncode if proc is not None else None
                    worker_status = json.load(open(status_path)) if os.path.exists(status_path) else {}
                    worker_peak = worker_status.get('peak_vram_gib')
                    error_type = worker_status.get('error_type') or (parent_failure or {}).get('error_type')
                    error_message = worker_status.get('error_message') or (parent_failure or {}).get('error_message')
                    manual_interrupt = error_type == 'KeyboardInterrupt'
                    explicit_failure = ((worker_returncode not in (None, 0) or parent_failure is not None)
                                        and not manual_interrupt)
                    done = (worker_status.get('completed') is True
                            and all(os.path.exists(path) for path in final_artifacts))
                    attempt.update({'end_step': end_step, 'ts_end': time.strftime('%Y-%m-%d %H:%M'),
                                    'elapsed_sec': round(wall, 1), 'completed': done,
                                    'peak_vram_gib': worker_peak, 'worker_returncode': worker_returncode,
                                    'error_type': error_type, 'error_message': error_message,
                                    'cuda_oom': bool(worker_status.get('cuda_oom', False)),
                                    'interrupted': manual_interrupt, 'resume_blocked': explicit_failure})
                    atomic_write_json(man_path, man)
                    gpu_row = gpu_snapshot()
                    if worker_peak is not None:
                        gpu_row['peak_vram_gb'] = worker_peak
                    append_ledger(LEDGER, {'run_tag': TAG,
                                           'condition': 'EXTERNAL_SFT_GRPO_train',
                                           'timings_sec': {'grpo_train': round(wall, 1)},
                                           'train_steps': end_step - start_step,
                                           'completed': attempt['completed'], 'gpu': gpu_row})
                if a2_concluded:
                    history = save_available_history()
                    evidence = {'num_generations': 4, 'history_records': len(history),
                                'last_logged_step': max((x.get('step', 0) for x in history), default=0)}
                    conclude_a2(RES_DIR, man['attempts'][-1], evidence=evidence)
                    print('concluded by sentinel a2 at the registered viability gate')
                else:
                    assert worker_status.get('completed') is True, 'worker status did not complete'
                    assert attempt['completed'] is True, 'final training artifacts are incomplete'
                    print(f'trained in {wall/3600:.2f}h, peak {worker_peak} GiB')
            else:
                print('adapter already trained')

            if concluded_a2_state(RES_DIR, man.get('attempts', [])) != 'committed':
                if training_state(RES_DIR, OUT_DIR, MERGED_DIR, cfg) == 'absent':
                    commit_training_complete(RES_DIR, OUT_DIR, MERGED_DIR, cfg)
                assert training_state(RES_DIR, OUT_DIR, MERGED_DIR, cfg) == 'committed'
                verify_archived_merge(ARCHIVE_MERGED, MERGED_DIR, ADIR)
                _, hashes_now = resolve_source_adapter(cfg.train.grpo_from_adapter, D)
                assert hashes_now == ADAPTER_HASHES, 'source adapter drifted since preflight'

                ep_state = endpoint_state(RES_DIR, OUT_DIR)
                if ep_state == 'tampered':
                    raise IntegrityError('endpoint files contradict their manifest')
                if ep_state != 'committed':
                    scratch = '/content/s30_eval_seed0'
                    _shutil.rmtree(scratch, ignore_errors=True)
                    t0 = time.time()
                    eval_done = False
                    try:
                        r = subprocess.run([sys.executable, EVAL_SCRIPT, '--config', CFG_PATH,
                                            '--adapter', OUT_DIR, '--out', scratch])
                        assert r.returncode == 0, 'endpoint evaluation failed'
                        eval_wall = round(time.time() - t0, 1)
                        gate_record = json.load(open(gate_path))
                        extra = {'adapter': OUT_DIR, 'source_adapter': ADIR,
                                 'source_adapter_sha256': ADAPTER_HASHES,
                                 'archived_merge': verify_archived_merge(
                                     ARCHIVE_MERGED, MERGED_DIR, ADIR),
                                 'parent_gate_stats': gate_record['stats'],
                                 'training_complete': json.load(open(
                                     os.path.join(RES_DIR, 'training_complete.json'))),
                                 'config': CFG_PATH, 'seed': 0, 'git': _git_info(),
                                 'versions': _versions(), 'resolved_dtype': resolve_dtype(cfg),
                                 'gpu': gpu_snapshot(), 'timings_sec': {'grpo_eval': eval_wall}}
                        commit_endpoint_eval(RES_DIR, scratch, adapter_dir=OUT_DIR,
                                             extra_manifest=extra, package='030_n4')
                        _shutil.rmtree(scratch, ignore_errors=True)
                        eval_done = True
                    finally:
                        append_ledger(LEDGER, {'run_tag': TAG,
                                               'condition': 'EXTERNAL_SFT_GRPO_eval',
                                               'timings_sec': {'grpo_eval': round(time.time() - t0, 1)},
                                               'train_steps': 0, 'completed': eval_done,
                                               'gpu': gpu_snapshot()})

                r = subprocess.run([sys.executable, 'scripts/run.py', '--config', CFG_PATH,
                                    '--stage', 'all', '--no-ledger'])
                assert r.returncode == 0, 'summary stage failed'
                summary = json.load(open(os.path.join(RES_DIR, 'summary.json')))
                assert summary['reference'] == f'gsm8k_sft_parent:{PARENT_TAG}'
                assert training_state(RES_DIR, OUT_DIR, MERGED_DIR, cfg) == 'committed'
                write_arm_complete(RES_DIR)
                print('gen per-tier:', summary['grpo_acc'])


Streaming output truncated to the last 5000 lines.
│ │ inside        │ assumptions   │               │              │           │ │
│ │ <answer></an… │ of Maslow's   │               │              │           │ │
│ │ tags.         │ hierarchy of  │               │              │           │ │
│ │               │ needs.</thin… │               │              │           │ │
│ │ Which of the  │ <answer>B</a… │               │              │           │ │
│ │ following is  │               │               │              │           │ │
│ │ an assumption │               │               │              │           │ │
│ │ in Maslow's   │               │               │              │           │ │
│ │ hierarchy of  │               │               │              │           │ │
│ │ needs?        │               │               │              │           │ │
│ │ A. Needs are  │               │               │              │           │ │
│ │ dependent on  │               │               │       

## 12. Registered comparison


In [14]:
import statistics
from analysis.stats import _read_preds, align_pred_rows, mcnemar_exact, paired_delta_ci
from data.subjects import TIERS, tier_of

def metric_summary(history, key):
    values = [(int(row['step']), float(row[key])) for row in history
              if row.get('step') is not None and row.get(key) is not None]
    if not values:
        return None
    first = [value for step, value in values if step <= 50]
    last = [value for step, value in values if step > 450]
    return {'mean': statistics.fmean(value for _, value in values),
            'first_50': statistics.fmean(first) if first else None,
            'last_50': statistics.fmean(last) if last else None}

def dynamics(path):
    history = json.load(open(path))
    keys = ('frac_reward_zero_std', 'reward_std',
            'rewards/accuracy_reward/mean', 'rewards/format_reward/mean',
            'kl', 'completions/mean_length', 'step_time')
    out = {key: metric_summary(history, key) for key in keys}
    out['total_generated_tokens'] = max(
        (float(row['num_tokens']) for row in history if row.get('num_tokens') is not None),
        default=None)
    out['logged_steps'] = max((int(row.get('step', 0)) for row in history), default=0)
    return out

comparison_path = os.path.join(D, 'results', '030_comparison_summary.json')
arm_manifest_path = os.path.join(RES_DIR, 'arm_manifest.json')
attempts = json.load(open(arm_manifest_path)).get('attempts', []) if os.path.exists(arm_manifest_path) else []
n8_history = os.path.join(PARENT_RESULTS, 'grpo_trainlog.json')
n4_history = os.path.join(RES_DIR, 'grpo_trainlog.json')

if concluded_a2_state(RES_DIR, attempts) == 'committed':
    result = {'experiment': '030', 'seed': 0, 'n4': 4, 'n8': 8,
              'status': 'viability_gate_failure', 'endpoint_category': None,
              'a2_record': json.load(open(os.path.join(RES_DIR, 'concluded_a2.json'))),
              'dynamics': {'n8': dynamics(n8_history),
                           'n4': dynamics(n4_history) if os.path.exists(n4_history) else None}}
    atomic_write_json(comparison_path, result)
    print('N=4 stopped at the registered a2 viability gate. No endpoint comparison was made.')
elif arm_state(RES_DIR, OUT_DIR) == 'committed':
    assert training_state(RES_DIR, OUT_DIR, MERGED_DIR, cfg) == 'committed'
    n8_rows = _read_preds(os.path.join(PARENT_RESULTS, 'grpo_preds.jsonl'))
    n4_rows = _read_preds(os.path.join(RES_DIR, 'grpo_preds.jsonl'))
    a8, a4 = align_pred_rows(n8_rows, n4_rows)

    tier_results = {}
    for tier in TIERS:
        pairs = [(old['correct'], new['correct']) for old, new in zip(a8, a4)
                 if tier_of(old['subject']) == tier]
        old_correct = [old for old, _ in pairs]
        new_correct = [new for _, new in pairs]
        delta, lo, hi = paired_delta_ci(old_correct, new_correct)
        tier_results[tier] = {'n8_acc': sum(old_correct) / len(old_correct),
                              'n4_acc': sum(new_correct) / len(new_correct),
                              'delta': delta, 'ci': [lo, hi]}

    hard_old = [old['correct'] for old, new in zip(a8, a4)
                if tier_of(old['subject']) == 'hard']
    hard_new = [new['correct'] for old, new in zip(a8, a4)
                if tier_of(old['subject']) == 'hard']
    hard_delta, hard_lo, hard_hi = paired_delta_ci(hard_old, hard_new)
    n01 = sum(1 for old, new in zip(hard_old, hard_new) if not old and new)
    n10 = sum(1 for old, new in zip(hard_old, hard_new) if old and not new)
    hard_p = mcnemar_exact(n01, n10)
    if abs(hard_delta) >= 0.02 and hard_p < 0.05:
        category = 'material improvement' if hard_delta > 0 else 'material decline'
    elif hard_lo > -0.02 and hard_hi < 0.02:
        category = 'equivalent'
    else:
        category = 'inconclusive'

    overall_old = [row['correct'] for row in a8]
    overall_new = [row['correct'] for row in a4]
    overall_delta, overall_lo, overall_hi = paired_delta_ci(overall_old, overall_new)
    tier_results['overall'] = {'n8_acc': sum(overall_old) / len(overall_old),
                               'n4_acc': sum(overall_new) / len(overall_new),
                               'delta': overall_delta, 'ci': [overall_lo, overall_hi]}

    p8 = _read_preds(os.path.join(PARENT_RESULTS, 'grpo_logprob_preds.jsonl'))
    p4 = _read_preds(os.path.join(RES_DIR, 'grpo_logprob_preds.jsonl'))
    pa8, pa4 = align_pred_rows(p8, p4)
    probe_delta, probe_lo, probe_hi = paired_delta_ci(
        [row['correct'] for row in pa8], [row['correct'] for row in pa4])
    probe_category = ('equivalent' if probe_lo > -0.02 and probe_hi < 0.02
                      else 'equivalence not established')

    format_n8 = sum(row.get('format_ok', False) for row in a8) / len(a8)
    format_n4 = sum(row.get('format_ok', False) for row in a4) / len(a4)
    length_n8 = statistics.fmean(row.get('n_tokens', 0) for row in a8)
    length_n4 = statistics.fmean(row.get('n_tokens', 0) for row in a4)
    n4_attempt = attempts[-1] if attempts else {}
    n8_manifest = json.load(open(os.path.join(PARENT_RESULTS, 'manifest.json')))

    result = {'experiment': '030', 'seed': 0, 'n4': 4, 'n8': 8,
              'status': 'completed', 'primary_endpoint': 'hard',
              'endpoint_category': category,
              'hard_mcnemar_p': hard_p, 'hard_discordant': {'n4_gain': n01, 'n8_gain': n10},
              'tiers': tier_results,
              'probe': {'delta': probe_delta, 'ci': [probe_lo, probe_hi],
                        'category': probe_category},
              'format_rate': {'n8': format_n8, 'n4': format_n4},
              'mean_response_tokens': {'n8': length_n8, 'n4': length_n4},
              'training_seconds': {'n8': n8_manifest.get('timings_sec', {}).get('grpo_train'),
                                   'n4': n4_attempt.get('elapsed_sec')},
              'peak_vram_gib': {'n8': n8_manifest.get('gpu', {}).get('peak_vram_gb'),
                                'n4': n4_attempt.get('peak_vram_gib')},
              'dynamics': {'n8': dynamics(n8_history), 'n4': dynamics(n4_history)},
              'scope': 'seed-0 matched artifact, fixed 500-step recipes'}
    atomic_write_json(comparison_path, result)
    print('hard:', f"N=8 {tier_results['hard']['n8_acc']:.3f}",
          f"N=4 {tier_results['hard']['n4_acc']:.3f}",
          f"delta {hard_delta:+.3f}", f"p {hard_p:.4g}", '->', category)
    print('overall delta:', f'{overall_delta:+.3f}',
          f'ci [{overall_lo:+.3f}, {overall_hi:+.3f}]')
    print('probe:', f'{probe_delta:+.3f}',
          f'ci [{probe_lo:+.3f}, {probe_hi:+.3f}]', '->', probe_category)
else:
    print('no completed or concluded arm yet')
print('comparison path:', comparison_path)


hard: N=8 0.639 N=4 0.636 delta -0.003 p 1 -> inconclusive
overall delta: -0.002 ci [-0.013, +0.009]
probe: -0.003 ci [-0.007, +0.002] -> equivalent
comparison path: /content/drive/MyDrive/inm363/results/030_comparison_summary.json


## 13. Peek at the result


In [15]:
comparison_path = os.path.join(D, 'results', '030_comparison_summary.json')
if os.path.exists(comparison_path):
    result = json.load(open(comparison_path))
    print('status:', result['status'])
    print('endpoint category:', result.get('endpoint_category'))
    if result['status'] == 'completed':
        print('hard:', result['tiers']['hard'])
        print('overall:', result['tiers']['overall'])
        print('probe:', result['probe'])
else:
    print('comparison not available yet')


status: completed
endpoint category: inconclusive
hard: {'n8_acc': 0.6388888888888888, 'n4_acc': 0.6363636363636364, 'delta': -0.0025252525252525255, 'ci': [-0.03787878787878788, 0.03282828282828283]}
overall: {'n8_acc': 0.8020169851380042, 'n4_acc': 0.7998938428874734, 'delta': -0.0021231422505307855, 'ci': [-0.01326963906581741, 0.009023354564755838]}
probe: {'delta': -0.002653927813163482, 'ci': [-0.006900212314225053, 0.0015923566878980893], 'category': 'equivalent'}


## 14. Close the units record


In [ ]:
# record the balance after the session finishes
import json, os, time
cu_after = 619.1     # fill from the resources panel
if cu_after is None:
    print('reminder: fill cu_after for the record')
elif not os.path.isdir('/content/drive/MyDrive'):
    print('drive not mounted, run the mount cell first')
else:
    os.makedirs(OPEN_UNITS, exist_ok=True)
    open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
    if not os.path.exists(open_path):
        print('no open record for this session, it may already be closed')
    else:
        with open(open_path) as f:
            record = json.load(f)
        record['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        record['colab_cu_after'] = cu_after
        record['colab_cu_used'] = round(record['colab_cu_before'] - cu_after, 2)
        record['closed_on_resume'] = False
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(record) + '\n')
        os.remove(open_path)
        print('logged:', record)


reminder: fill cu_after for the record
